# Connector ও Visual Token Compression

Connectors and visual token compression — N টি vision token-কে LLM-এর জন্য K < N টি token-এ পরিণত করার চারটি উপায়, scratch থেকে train করা এবং সমান K-তে তুলনা করা।

কোনো download নেই, কোনো pretrained weight নেই। CPU-তে কয়েক মিনিট।

**কাজটি** হলো Lesson 3-এর grounding কাজ, বড় করে: 16টি বস্তুর একটি scene, প্রতিটির একটি করে vision token (shape + color), এবং একটি প্রশ্ন যা একটি shape-এর নাম বলে। Model-কে সেই বস্তুর color দিয়ে উত্তর দিতে হবে। Chance = 1/6 = 16.7%।

**তুলনা করা connector-গুলো**, সবগুলোই (16টি vision token) -> (K টি token) mapping করে:

1. **MLP projector** (LLaVA) — `K = 16`, কোনো compression-ই নেই।
2. **Average pooling** — পাশাপাশি token-দের K টি group, গড় করা।
3. **Learned-query resampler** (Perceiver / Q-Former, BLIP-2) — K টি learned query vision token-গুলোতে cross-attend করে; query-গুলো প্রশ্নটি **দেখে না**।
4. **Instruction-aware resampler** (InstructBLIP-style) — একই, তবে query-গুলো আগে প্রশ্নের উপর condition করা হয়।

যে পরিমাপটি গুরুত্বপূর্ণ: K যত ছোট হয়, accuracy কেমন থাকে। একটি query-agnostic compressor-কে অনুমান করতে হয় কী জিজ্ঞাসা করা হবে; একটি query-aware compressor ঠিক সেই একটি বস্তু রেখে দিতে পারে যার কথা প্রশ্নে আছে। এই পার্থক্য `K = 16`-এ অদৃশ্য এবং `K = 1`-এ নির্ণায়ক।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান (CPU-তে কয়েক মিনিট; বেশিরভাগ সময় যায় Section 3-এর training table-এ)। Section 1 ও 2 কেবল connector, LLM ও training function সংজ্ঞায়িত করে; Section 5 Section 3-এ তৈরি `table` ব্যবহার করে।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

NUM_SHAPES = 20
NUM_COLORS = 6
NUM_OBJECTS = 16          # "tower"-এর তৈরি vision token-সংখ্যা
D_VISION = 32
D_MODEL = 64
N_TEXT = 4

## 0. Synthetic scene ও প্রশ্ন

Lesson 3-এর মতোই: প্রতিটি shape ও color-এর একটি স্থির random code, আর একটি বস্তুর vision token হলো দুটোর যোগফল যোগ noise। `make_batch` প্রতিটি scene-এ 16টি ভিন্ন shape রাখে, একটি বস্তু বেছে নিয়ে প্রশ্ন `[WHAT] [COLOR] [IS] [<shape>]` তৈরি করে এবং উত্তর হিসেবে তার color id ফেরত দেয়।

In [ ]:
shape_code = torch.randn(NUM_SHAPES, D_VISION)
color_code = torch.randn(NUM_COLORS, D_VISION)
TEXT_VOCAB = 3 + NUM_SHAPES


def make_batch(batch_size):
    shapes = torch.stack([torch.randperm(NUM_SHAPES)[:NUM_OBJECTS] for _ in range(batch_size)])
    colors = torch.randint(0, NUM_COLORS, (batch_size, NUM_OBJECTS))
    vision = shape_code[shapes] + color_code[colors]
    vision = vision + 0.2 * torch.randn_like(vision)
    slot = torch.randint(0, NUM_OBJECTS, (batch_size,))
    queried = shapes[torch.arange(batch_size), slot]
    answer = colors[torch.arange(batch_size), slot]
    fixed = torch.tensor([0, 1, 2]).expand(batch_size, 3)
    question = torch.cat([fixed, (queried + 3).unsqueeze(1)], dim=1)
    return vision, question, answer

## 1. চারটি connector

প্রতিটি connector একই interface মেনে চলে: `forward(vision, question_embed)` নেয় এবং LLM-এর জন্য `(B, K, D_MODEL)` আকারের token ফেরত দেয়।

- `MLPProjector` — LLaVA-র connector: প্রতি token-এ একটি MLP, `K = N`।
- `PoolProjector` — পাশাপাশি token-গুলোকে K টি group-এ গড় করে তারপর project করে; spatially local ও content-blind।
- `QueryResampler` — K টি learned query vision token-এ cross-attend করে। `instruction_aware=True` হলে query-গুলোতে আগে প্রশ্নের embedding-এর গড় যোগ করা হয় (InstructBLIP-এর পরিবর্তন)।

`CONNECTORS` dict প্রতিটি নামকে একটি factory function-এর সাথে যুক্ত করে যা `k` নিয়ে connector তৈরি করে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. চারটি connector
# ---------------------------------------------------------------------------

class MLPProjector(nn.Module):
    """LLaVA-র connector: প্রতি token-এ একটি MLP, K = N। কিছুই compress হয় না।"""

    def __init__(self, k):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D_VISION, D_MODEL), nn.GELU(),
                                 nn.Linear(D_MODEL, D_MODEL))

    def forward(self, vision, question_embed):
        return self.net(vision)                                   # (B, N, D_MODEL)


class PoolProjector(nn.Module):
    """পাশাপাশি token-গুলোকে K টি group-এ গড় করো, তারপর project করো।

    এটি সস্তা, parameter-free compressor -- যে family-তে pixel-shuffle /
    patch-merge কৌশলগুলোও পড়ে। এটি spatially local: image বা প্রশ্ন যা-ই হোক,
    group g সবসময় একই region-এর সারসংক্ষেপ করে।
    """

    def __init__(self, k):
        super().__init__()
        self.k = k
        self.net = nn.Sequential(nn.Linear(D_VISION, D_MODEL), nn.GELU(),
                                 nn.Linear(D_MODEL, D_MODEL))

    def forward(self, vision, question_embed):
        b, n, d = vision.shape
        pooled = vision.reshape(b, self.k, n // self.k, d).mean(dim=2)
        return self.net(pooled)


class QueryResampler(nn.Module):
    """Perceiver Resampler / Q-Former: K টি LEARNED query vector vision
    token-গুলোতে cross-attend করে; তাদের output-ই compressed sequence।

    Query-গুলো parameter -- প্রতিটি image-কে একই K টি প্রশ্ন করা হয়, এবং
    গুরুত্বপূর্ণভাবে সেগুলো বেছে নেওয়া হয় user কী জিজ্ঞাসা করবে তা কেউ
    জানার আগেই।
    """

    def __init__(self, k, instruction_aware=False):
        super().__init__()
        self.k = k
        self.instruction_aware = instruction_aware
        self.queries = nn.Parameter(torch.randn(1, k, D_MODEL) * 0.02)
        self.kv_proj = nn.Linear(D_VISION, D_MODEL)
        self.ln_q = nn.LayerNorm(D_MODEL)
        self.ln_kv = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, 4, batch_first=True)
        self.ff = nn.Sequential(nn.LayerNorm(D_MODEL), nn.Linear(D_MODEL, 2 * D_MODEL),
                                nn.GELU(), nn.Linear(2 * D_MODEL, D_MODEL))

    def forward(self, vision, question_embed):
        b = vision.shape[0]
        q = self.queries.expand(b, -1, -1)
        if self.instruction_aware:
            # InstructBLIP-এর পরিবর্তন: image থেকে কী বের করবে তা ঠিক করার
            # আগে query-গুলোকে instruction দেখতে দাও।
            q = q + question_embed.mean(dim=1, keepdim=True)
        kv = self.ln_kv(self.kv_proj(vision))
        out = self.attn(self.ln_q(q), kv, kv, need_weights=False)[0]
        out = q + out
        return out + self.ff(out)


CONNECTORS = {
    "MLP projector (no compression)": lambda k: MLPProjector(k),
    "average pooling": lambda k: PoolProjector(k),
    "learned-query resampler": lambda k: QueryResampler(k, instruction_aware=False),
    "instruction-aware resampler": lambda k: QueryResampler(k, instruction_aware=True),
}

## 2. যে (ক্ষুদ্র) LLM-কে connector খাওয়ায়

`VLM` একটি 2-layer Transformer: প্রশ্নকে embed করে, connector-এর মাধ্যমে vision token তৈরি করে (connector প্রশ্নের embedding-ও পায়, যদিও কেবল instruction-aware resampler সেটি ব্যবহার করে), দুটোকে একটি sequence-এ concatenate করে এবং শেষ token থেকে color predict করে। Connector ছাড়া downstream-এর সবকিছু প্রতিটি পরীক্ষায় হুবহু এক।

`train_and_eval` একটি connector ও `k`-এর জন্য model-টি 700 step train করে, 2000টি held-out scene-এ accuracy মাপে এবং connector-এর parameter-সংখ্যা ফেরত দেয়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. যে (ক্ষুদ্র) LLM-কে connector খাওয়ায়
# ---------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, 4, batch_first=True)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.ff = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                nn.Linear(4 * D_MODEL, D_MODEL))

    def forward(self, x):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.ff(self.ln2(x))


class VLM(nn.Module):
    def __init__(self, connector_fn, k, n_layers=2):
        super().__init__()
        self.tok_embed = nn.Embedding(TEXT_VOCAB, D_MODEL)
        self.connector = connector_fn(k)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, D_MODEL) * 0.02)
        self.blocks = nn.ModuleList([Block() for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, NUM_COLORS)

    def forward(self, vision, question):
        txt = self.tok_embed(question)
        vis = self.connector(vision, txt)
        x = torch.cat([vis, txt], dim=1)
        x = x + self.pos_embed[:, : x.shape[1]]
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x[:, -1]))


def train_and_eval(connector_fn, k, steps=700, batch_size=64, lr=2e-3):
    torch.manual_seed(1)
    model = VLM(connector_fn, k)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        vision, question, answer = make_batch(batch_size)
        loss = F.cross_entropy(model(vision, question), answer)
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        vision, question, answer = make_batch(2000)
        acc = (model(vision, question).argmax(1) == answer).float().mean().item()
    n_conn = sum(p.numel() for p in model.connector.parameters())
    return acc, n_conn

## 3. পরিমাপ: K কমার সাথে সাথে accuracy

এই cell প্রতিটি connector-কে `K = 16, 8, 4, 1`-এ আলাদাভাবে train করে এবং একটি accuracy table ছাপে। MLP projector token-সংখ্যা বদলাতে পারে না, তাই সেটি কেবল `K = 16`-এ চলে (বাকি ঘরে `--`)। ফলাফল ও connector parameter-সংখ্যা `table` dict-এ রাখা হয়, যা Section 5 ব্যবহার করে। এরপর table-টি কীভাবে পড়তে হবে তার ব্যাখ্যা ছাপা হয়: pooling ধীরে ধীরে ভাঙে, learned-query resampler ভালো করে কিন্তু `K = 1`-এ একই দেয়ালে ঠেকে, আর instruction-aware resampler প্রশ্ন আগে দেখে বলে `K = 1`-এও টিকে থাকে।

In [ ]:
print("=" * 78)
print("TASK: 16 objects (1 vision token each), 'what color is the <shape>?'")
print(f"Chance = {1 / NUM_COLORS:.1%}. Every connector feeds the same 2-layer LLM.")
print("=" * 78)
print()
print("Accuracy as the connector is forced to emit fewer tokens (K):")
print()
header = f"{'connector':<32}" + "".join(f"{'K=' + str(k):>10}" for k in [16, 8, 4, 1])
print(header)
print("-" * len(header))

table = {}
for name, fn in CONNECTORS.items():
    row = []
    for k in [16, 8, 4, 1]:
        if name.startswith("MLP") and k != NUM_OBJECTS:
            row.append(None)                    # একটি MLP token-সংখ্যা বদলাতে পারে না
            continue
        acc, n_conn = train_and_eval(fn, k)
        row.append(acc)
        table[(name, k)] = (acc, n_conn)
    cells = "".join(f"{'  --':>10}" if a is None else f"{a:>9.1%}" for a in row)
    print(f"{name:<32}{cells}")

print()
print("Reading the table:")
print()
print("* At K=16 nothing is being compressed and everything works. All the")
print("  interesting behaviour is in the columns to the right.")
print()
print("* Average pooling degrades steadily. It is spatially local and content-")
print("  blind: group g always averages the same slots, so as groups get bigger")
print("  the objects inside one group blur together and the shape->color binding")
print("  is lost -- Lesson 3's pooled baseline, arriving gradually.")
print()
print("* The learned-query resampler compresses adaptively (its queries can")
print("  learn to attend to whichever slots are informative) but its queries are")
print("  FIXED PARAMETERS: the same K questions are asked of every image, chosen")
print("  before the user's question is known. At K=1 that is a hard information")
print("  bottleneck -- one vector cannot carry 16 shape->color bindings.")
print()
print("* The instruction-aware resampler sees the question before it compresses,")
print("  so at K=1 it only has to keep the ONE binding that was asked about. Same")
print("  parameter count, same K, radically different accuracy.")
print()

## 4. Compression আসলে কী সাশ্রয় করে

Lesson 1-এর বাস্তব token-সংখ্যা ব্যবহার করে (576টি token @336px, 3,136টি AnyRes 896px-এ), এই cell একটি 100-token prompt সহ vision token-গুলোর উপর LLM self-attention-এর একটি layer-এ কতগুলো attention cell লাগে তা গণনা করে, এবং uncompressed CLIP ViT-L/14 @336-এর তুলনায় সাশ্রয় দেখায়। শেষে সতর্কবার্তা: প্রশ্ন না জেনে বেছে নেওয়া compressor কেবল *সাধারণত* যা কাজে লাগে তা-ই রাখতে পারে।

In [ ]:
print("=" * 78)
print("WHAT THE COMPRESSION ACTUALLY BUYS (real numbers from Lesson 1)")
print("=" * 78)
print()
print(f"{'setting':<40} {'vis tokens':>11} {'LLM attn cells':>16} {'saving':>9}")
prompt_len = 100
base = None
for label, n_vis in [("CLIP ViT-L/14 @336, no compression", 576),
                     ("2x2 pixel-shuffle (4x fewer)", 144),
                     ("resampler to 64 queries", 64),
                     ("resampler to 32 queries", 32),
                     ("AnyRes 896px, no compression", 3136)]:
    total = n_vis + prompt_len
    cells = total * total
    if base is None:
        base = cells
    print(f"{label:<40} {n_vis:>11,} {cells:>16,} {base / cells:>8.1f}x")
print()
print(f"(one layer of self-attention over vision tokens + a {prompt_len}-token prompt)")
print()
print("Those savings are why every production VLM compresses somewhere. But the")
print("table above is the warning that comes with them: a compressor chosen")
print("without knowing the question can only keep what is USUALLY useful, and")
print("the queries it drops are gone before the LLM ever sees the image. That is")
print("the whole trade -- context budget against the risk that the one detail the")
print("user asked about was in the discarded 90%.")
print()

## 5. Connector-এর parameter-সংখ্যা (K=4)

Section 3-এর `table` থেকে `K = 4`-এ প্রতিটি connector-এর parameter-সংখ্যা ও accuracy দেখানো হয় (MLP projector `K = 4`-এ চলেনি, তাই সেটি বাদ পড়ে)। মূল কথা: যেটিই বেছে নিন, connector-গুলো parameter-এ সস্তা — পছন্দটি model-এর আকার নিয়ে নয়, বরং কোন তথ্য টিকে থাকে তা নিয়ে।

In [ ]:
print("=" * 78)
print("CONNECTOR PARAMETER COUNTS (K=4)")
print("=" * 78)
for name in CONNECTORS:
    if (name, 4) in table:
        acc, n_conn = table[(name, 4)]
        print(f"{name:<32} {n_conn:>8,} params   acc {acc:>6.1%}")
print()
print("Connectors are cheap in parameters no matter which you pick -- the choice")
print("is about what information survives them, not about model size.")